# HuecoEnv: GRPO Training Loop

This notebook contains the complete TRL (Transformer Reinforcement Learning) training loop for the HuecoEnv multi-agent simulation using GRPO. It adapts the Meta OpenEnv standard architecture to train an LLM to navigate the Scarcity Droughts.

In [ ]:
# Install training dependencies for custom rollout mode.
# We remove stale packages first, then install a known-compatible vLLM.
%pip uninstall -y vllm trackio
%pip install -U git+https://github.com/huggingface/trl.git git+https://github.com/meta-pytorch/OpenEnv.git bitsandbytes
%pip install --no-cache-dir --force-reinstall "vllm==0.17.1"

# Sanity-check vLLM import in THIS kernel so failures show up immediately.
import importlib
import traceback

try:
    vllm = importlib.import_module("vllm")
    print("vllm import OK:", vllm.__version__)
except Exception:
    print("vllm import failed. Fix install before continuing.")
    traceback.print_exc()

print("If packages changed, restart the kernel before running the next cell.")

In [ ]:
import os
import importlib
from huggingface_hub import login

# Runtime flags: silence experimental warning and force legacy vLLM engine
# to avoid torch.compile backend issues in some HF images.
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"
os.environ["VLLM_USE_V1"] = "0"

# Hard fail early if vLLM is unavailable.
vllm = importlib.import_module("vllm")
print("Using vLLM:", vllm.__version__)

# Use HF_TOKEN env var when available and avoid git credential helper warnings.
if os.getenv("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
else:
    login(add_to_git_credential=False)

## Initialize the Environment

We initialize the `HuecoGymWrapper` which provides the standard OpenAI Gym API for our custom multi-agent logic.

In [ ]:
from train import HuecoGymWrapper

# We train specifically on the hard task so the agent experiences the Injector
task_name = "adaptive_survival"
wrapper = HuecoGymWrapper(task_name=task_name)

## Initialize Model and Tokenizer

We use a lightweight model for the hackathon baseline. You can scale this up to Llama-3.1-8B-Instruct if you have sufficient GPU memory.

In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

## GRPO Configuration

We configure Group Relative Policy Optimization with compatibility guards across TRL versions. Custom OpenEnv rollouts use `generate_rollout_completions`, so vLLM is enabled with a supported version.

In [ ]:
from trl import GRPOConfig
from datasets import Dataset

# Create a dummy dataset (GRPO requires prompts to start generation).
dataset = Dataset.from_dict({"prompt": ["Solve the environment."] * 500})

prompt_len = 1400
completion_len = 128

# Custom rollouts with generate_rollout_completions require vLLM.
use_vllm = True

cfg = {
    "num_train_epochs": 1,
    "learning_rate": 5e-6,
    "gradient_accumulation_steps": 8,
    "per_device_train_batch_size": 1,
    "warmup_steps": 20,
    "num_generations": 2,
    "max_completion_length": completion_len,
    "use_vllm": use_vllm,
    "output_dir": "huecoenv-grpo",
    "logging_steps": 1,
    "save_steps": 10,
}

if use_vllm:
    cfg["vllm_mode"] = "colocate"
    cfg["vllm_gpu_memory_utilization"] = 0.3

# Handle length argument differences across TRL versions.
valid = set(getattr(GRPOConfig, "__dataclass_fields__", {}).keys())
if "max_prompt_length" in valid:
    cfg["max_prompt_length"] = prompt_len
elif "max_seq_length" in valid:
    cfg["max_seq_length"] = prompt_len + completion_len
elif "max_length" in valid:
    cfg["max_length"] = prompt_len + completion_len

cfg = {k: v for k, v in cfg.items() if k in valid}
grpo_config = GRPOConfig(**cfg)
print("GRPO config keys:", sorted(cfg.keys()))

## Training Loop & Rollout Function

The rollout function defines how the agent interacts with the environment during GRPO training.

In [ ]:
from trl import GRPOTrainer
from trl.experimental.openenv import generate_rollout_completions
import json
import csv

# To track our survival rate over time
training_log = []

def rollout_func(prompts, trainer=None):
    episode_prompt_ids = []
    episode_completion_ids = []
    episode_logprobs = []
    correctness_rewards = []

    for prompt_text in prompts:
        obs = wrapper.reset()
        done = False
        total_reward = 0.0
        
        # Generate actions using the model
        rollout_outputs = generate_rollout_completions(trainer, [prompt_text])[0]
        
        # In a real run, this parses the JSON action from the LLM and steps the environment
        try:
            action_text = rollout_outputs.get("text")
            obs, reward, done, info = wrapper.step()
            total_reward += reward
        except Exception:
            total_reward -= 1.0 # Penalty for bad formatting
            
        # Record the survival rate from the Environment Brain
        brain_info = wrapper.end_episode()
        survival_rate = brain_info.get("survival_rate", 0.0)
        training_log.append({"survival_rate": survival_rate})
        
        episode_prompt_ids.append(rollout_outputs["prompt_ids"])
        episode_completion_ids.append(rollout_outputs["completion_ids"])
        episode_logprobs.append(rollout_outputs["logprobs"])
        correctness_rewards.append(total_reward)

    return {
        "prompt_ids": episode_prompt_ids,
        "completion_ids": episode_completion_ids,
        "logprobs": episode_logprobs,
        "correct_reward": correctness_rewards,
    }

def reward_correct(completions, **kwargs):
    rewards = kwargs.get("correct_reward")
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

trainer = GRPOTrainer(
    model=model_name,
    processing_class=tokenizer,
    reward_funcs=[reward_correct],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
)

print("Starting TRL GRPO Training...")
trainer.train()

# Save data for the final graph
os.makedirs("data", exist_ok=True)
with open("data/training_adaptive_survival.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["episode", "survival_rate"])
    for idx, row in enumerate(training_log):
        writer.writerow([idx + 1, row["survival_rate"]])
        
print("Training Complete! Graph data saved.")

In [ ]:
# Plot comparison curves from saved CSV files
import pandas as pd
import matplotlib.pyplot as plt

train_path = "data/training_adaptive_survival.csv"
sim_path = "data/simulation_adaptive_survival.csv"

train_df = pd.read_csv(train_path)
sim_df = pd.read_csv(sim_path)

# Normalize schema in case files come with extra columns.
train_df = train_df[["episode", "survival_rate"]].copy()
sim_df = sim_df[["episode", "survival_rate"]].copy()

# Rolling means help readability when curves are noisy.
train_df["survival_rate_smooth"] = train_df["survival_rate"].rolling(window=25, min_periods=1).mean()
sim_df["survival_rate_smooth"] = sim_df["survival_rate"].rolling(window=10, min_periods=1).mean()

plt.figure(figsize=(11, 6))

# Raw curves (light)
plt.plot(train_df["episode"], train_df["survival_rate"], alpha=0.2, linewidth=1.0, label="Training raw")
plt.plot(sim_df["episode"], sim_df["survival_rate"], alpha=0.3, linewidth=1.0, label="Simulation raw")

# Smoothed curves (bold)
plt.plot(train_df["episode"], train_df["survival_rate_smooth"], linewidth=2.5, label="Training (rolling mean)")
plt.plot(sim_df["episode"], sim_df["survival_rate_smooth"], linewidth=2.5, label="Simulation (rolling mean)")

plt.title("Adaptive Survival Rate Curves")
plt.xlabel("Episode")
plt.ylabel("Survival Rate")
plt.ylim(0.0, 1.05)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

out_path = "data/adaptive_survival_curves.png"
plt.savefig(out_path, dpi=180)
plt.show()

print(f"Saved plot to: {out_path}")